In [1]:
!pip install -q tiktoken

In [2]:
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1"
)

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1/"
    "gpt2_wikitext_300k.pt"
)

# Save the new 300K checkpoint in Kaggle working storage
OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_400k.pt"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Resume checkpoint:", CHECKPOINT_PATH)
print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1
Resume checkpoint: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_300k.pt
Output checkpoint: /kaggle/working/gpt2_wikitext_400k.pt
Device: cuda:0
GPU: Tesla T4


In [3]:
print("Resume checkpoint:", CHECKPOINT_PATH)
print("Output checkpoint:", OUTPUT_CHECKPOINT)

Resume checkpoint: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_300k.pt
Output checkpoint: /kaggle/working/gpt2_wikitext_400k.pt


In [4]:
required_files = [
    PROJECT_ROOT / "src" / "llm" / "gpt" / "model.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "trainer.py",
    PROJECT_ROOT / "data" / "wikitext-103" / "train.txt",
    PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt",
    CHECKPOINT_PATH,
]

for file_path in required_files:
    print(file_path, "->", file_path.exists())

if not all(file_path.exists() for file_path in required_files):
    raise FileNotFoundError(
        "One or more required files are missing."
    )

/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/src/llm/gpt/model.py -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/src/llm/training/trainer.py -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/data/wikitext-103/train.txt -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/data/wikitext-103/validation.txt -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_300k.pt -> True


In [5]:
train_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "train.txt"
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "validation.txt"
)

train_text = train_path.read_text(
    encoding="utf-8"
)

validation_text = validation_path.read_text(
    encoding="utf-8"
)

print("Training characters:", len(train_text))
print("Validation characters:", len(validation_text))

Training characters: 540624389
Validation characters: 1147070


In [6]:
import tiktoken
import torch

from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

BATCH_SIZE = 1
CONTEXT_LENGTH = 1024

train_loader = create_data_loader(
    text=train_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=True,
    drop_last=True,
    num_workers=2,
)

validation_loader = create_data_loader(
    text=validation_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
    drop_last=False,
    num_workers=2,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Effective batch size:", BATCH_SIZE)
print("Context length:", CONTEXT_LENGTH)
print("CUDA available:", torch.cuda.is_available())

Training batches: 116294
Validation batches: 243
Effective batch size: 1
Context length: 1024
CUDA available: True


In [7]:
from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = CONTEXT_LENGTH

model = GPTModel(config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.1,
)

print("Model created.")
print("Parameters:", sum(
    parameter.numel()
    for parameter in model.parameters()
))

Model created.
Parameters: 163037184


In [8]:
from llm.training.trainer import train_model_simple

print("Training function imported.")

Training function imported.


In [9]:
variables_to_check = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "OUTPUT_CHECKPOINT": OUTPUT_CHECKPOINT,
    "train_loader": train_loader,
    "validation_loader": validation_loader,
    "tokenizer": tokenizer,
    "model": model,
    "optimizer": optimizer,
    "device": device,
    "config": config,
}

for name, value in variables_to_check.items():
    print(name, "-> OK" if value is not None else "MISSING")

print("Input checkpoint exists:", CHECKPOINT_PATH.exists())

PROJECT_ROOT -> OK
CHECKPOINT_PATH -> OK
OUTPUT_CHECKPOINT -> OK
train_loader -> OK
validation_loader -> OK
tokenizer -> OK
model -> OK
optimizer -> OK
device -> OK
config -> OK
Input checkpoint exists: True


In [10]:
train_losses, validation_losses, tokens_seen = (
    train_model_simple(
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=100,
        eval_freq=500,
        eval_iter=5,
        start_context="The history of the world",
        tokenizer=tokenizer,
        checkpoint_path=str(OUTPUT_CHECKPOINT),
        config=config,
        max_steps=400_000,
        resume_from_checkpoint=str(CHECKPOINT_PATH),
    )
)

Resumed step 300000, epoch 9, batch 83707
Legacy checkpoint has no scheduler state; schedule starts from its restored optimizer LR.
AMP enabled: FP16
Step 300500: train 2.7572, val 2.5079, ppl 12.279, lr 1e-05
Step 301000: train 2.5127, val 2.5052, ppl 12.245, lr 1e-05
Step 301500: train 2.7398, val 2.5042, ppl 12.234, lr 1e-05
Step 302000: train 2.6379, val 2.5058, ppl 12.253, lr 1e-05
Step 302500: train 2.8102, val 2.5093, ppl 12.296, lr 1e-05
Step 303000: train 3.0179, val 2.5071, ppl 12.270, lr 1e-05
Step 303500: train 2.7009, val 2.5145, ppl 12.361, lr 1e-05
Step 304000: train 2.9984, val 2.5099, ppl 12.304, lr 1e-05
Step 304500: train 2.8704, val 2.5019, ppl 12.206, lr 1e-05
Step 305000: train 2.5487, val 2.5033, ppl 12.223, lr 1e-05
Step 305500: train 2.6992, val 2.5058, ppl 12.253, lr 1e-05
Step 306000: train 2.8155, val 2.5029, ppl 12.217, lr 1e-05
Step 306500: train 2.8583, val 2.5075, ppl 12.274, lr 1e-05
Step 307000: train 2.5963, val 2.5015, ppl 12.200, lr 1e-05
Step 30750

In [11]:
if not OUTPUT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint was not created: {OUTPUT_CHECKPOINT}"
    )

print("Checkpoint exists:", OUTPUT_CHECKPOINT.exists())
print(
    "Checkpoint size:",
    round(OUTPUT_CHECKPOINT.stat().st_size / 1024**3, 2),
    "GB",
)

final_checkpoint = torch.load(
    OUTPUT_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("Final global step:", final_checkpoint["global_step"])
print("Final epoch:", final_checkpoint["epoch"])
print("Final training loss:", final_checkpoint["train_loss"])
print("Final validation loss:", final_checkpoint["val_loss"])

Checkpoint exists: True
Checkpoint size: 1.87 GB
Final global step: 400000
Final epoch: 10
Final training loss: 3.3599019050598145
Final validation loss: 2.4788486480712892
